# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliAlmasri7/FlyRank-ML/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

### Problem framing

The W04 baseline is a rule-based prioritization system. It identifies content whose observed CTR is below the aggregated CTR of pages in the same position group, while giving additional priority to pages with more impressions.

For W05, I want to test whether a machine-learning model can use signals available in the observation period to identify content associated with a future performance outcome.

I will use:

- March 2026 as the feature/observation period.
- April 2026 as the future evaluation period.

This prevents the model from using future information when making its prediction.

### Model choice

I will use **Random Forest Regression**.

The target is a continuous future-performance measure, so regression is appropriate.

Random Forest is useful for this experiment because:

- it can model non-linear relationships;
- it can handle interactions between signals;
- it does not require feature scaling;
- feature importance can be inspected;
- it provides a useful comparison against the simple W04 rule.

The goal is not to maximize model complexity. The goal is to determine whether the model provides useful predictive information beyond the W04 baseline.

### Important leakage consideration

The W04 `ctr_gap` should not be used as a model feature.

It is directly constructed from the observed CTR and the position-group benchmark used by the baseline. Including it would make the model too closely reproduce the baseline rather than provide an independent comparison.

Likewise, future April performance variables must not be used as input features.

In [1]:
import duckdb
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr

import matplotlib.pyplot as plt

### Data source

The model uses the same FlyRank warehouse table as W04:

`fact_content_daily_performance`

The March period provides the features, while April provides the future outcome.

## 2. Split design

I use a chronological split rather than a random train/test split.

### Feature period

March 2026

The model only sees information available during March:

- GSC impressions
- GSC clicks
- GSC position
- derived CTR

### Future evaluation period

April 2026

The model is evaluated against performance observed after the feature period.

This is preferable to randomly splitting March rows because random splitting could allow observations from the same content item and time period to appear in both training and validation data.

The temporal setup better matches the intended use case:

> use current information to prioritize content for future review.

In [6]:
import os 
# Connect to DuckDB
con = duckdb.connect()

# Hugging Face token
HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN was not found in the environment.")

# Create Hugging Face secret
con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

# Dataset location
rel = "hf://datasets/FlyRank/internship-warehouse"

# March 2026 partition
march_path = (
    f"{rel}/fact_content_daily_performance/"
    f"month=2026-03/*.parquet"
)

print("Dataset connected successfully.")
print("Using:", march_path)

Dataset connected successfully.
Using: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet


In [8]:
march_df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    SUM(gsc_sum_position) AS gsc_sum_position
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-03/*.parquet'
)
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

march_df["observed_ctr"] = (
    march_df["gsc_clicks"]
    / march_df["gsc_impressions"].replace(0, np.nan)
)

march_df["avg_position"] = (
    march_df["gsc_sum_position"]
    / march_df["gsc_impressions"].replace(0, np.nan)
)

march_df.head()

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,observed_ctr,avg_position
0,client_62f4a7e64f5e0096,content_42178e650f9d310b,12.0,0.0,238.0,0.000000,19.833333
1,client_62f4a7e64f5e0096,content_ca8ddf870fb58b51,205.0,0.0,2320.0,0.000000,11.317073
2,client_62f4a7e64f5e0096,content_dae518ae539dcca9,173.0,0.0,1985.0,0.000000,11.473988
3,client_62f4a7e64f5e0096,content_b9b75ec457ee78c7,10111.0,16.0,9242.0,0.001582,0.914054
4,client_62f4a7e64f5e0096,content_56fc793443887c43,1756.0,6.0,3713.0,0.003417,2.114465


In [18]:
april_df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS future_impressions,
    SUM(gsc_clicks) AS future_clicks
FROM read_parquet(
    '{rel}/fact_content_daily_performance/month=2026-04/*.parquet'
)
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

april_df["future_ctr"] = (
    april_df["future_clicks"]
    / april_df["future_impressions"].replace(0, np.nan)
)

april_df.head()

,client_hash_id,content_hash_id,future_impressions,future_clicks,future_ctr
0,client_62f4a7e64f5e0096,content_49dcc118cfb555d0,2922.0,0.0,0.000000
1,client_62f4a7e64f5e0096,content_d02cd6961c2fa131,554.0,0.0,0.000000
2,client_62f4a7e64f5e0096,content_0852b0b54c54cf84,6499.0,1.0,0.000154
3,client_62f4a7e64f5e0096,content_71fe97b5578b0a8b,12.0,0.0,0.000000
4,client_62f4a7e64f5e0096,content_0ae19ba5aeb0ecc0,1574.0,1.0,0.000635


In [19]:
model_df = march_df.merge(
    april_df,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

model_df.shape

(331436, 10)

In [20]:
print("Rows with March + April data:", len(model_df))

print(
    "Future CTR missing:",
    model_df["future_ctr"].isna().sum()
)

print(
    "Future CTR available:",
    model_df["future_ctr"].notna().sum()
)

print(
    "Future CTR summary:"
)

model_df["future_ctr"].describe()

Rows with March + April data: 331436
Future CTR missing: 154995
Future CTR available: 176441
Future CTR summary:


count    176441.000000
mean          0.003292
std           0.030496
min           0.000000
25%           0.000000
50%           0.000000
75%           0.001623
max           1.000000
Name: future_ctr, dtype: float64

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.